# Extra Experiment: Artist/Album Hierarchical R-KMeans

Этот notebook добавляет hard metadata-anchored fixed-length SID baseline:

```text
sid[0] = artist cluster
sid[1] = album cluster внутри artist cluster
sid[2:] = residual k-means коды по трековым embedding'ам
```

Кластеры artist/album строятся по средним item embeddings. Это не dVAE и не soft loss, а интерпретируемая иерархическая baseline-модель: `artist -> album -> track residual`.

Зачем это полезно:

- сравнить learned prefix supervision с жестко заданной metadata hierarchy;
- проверить, помогает ли музыкальная структура сама по себе;
- получить очень понятный fixed-length baseline для отчета.

In [ ]:
from collections import defaultdict
from pathlib import Path
import copy
import json
import subprocess
import sys

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
import yaml

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run this notebook from the repository root")

PY = sys.executable
DATA_DIR = ROOT / "data" / "RQ_album_artist_anchor" / "yambda"
RESULTS_DIR = ROOT / "results" / "RQ_album_artist_anchor"
CONFIG_DIR = ROOT / "configs" / "RQ_album_artist_anchor"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def run(cmd, *, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=ROOT, check=check)

DATA_DIR, RESULTS_DIR, DEVICE

## Experiment Grid

По умолчанию запускаем один основной вариант. Если есть время, можно добавить второй вариант с большим artist codebook.

Параметры:

- `artist_k`: число кластеров артистов;
- `album_k`: максимум album clusters внутри каждого artist cluster;
- `residual_levels`: сколько residual k-means уровней после artist/album;
- `residual_k`: число кластеров на каждом residual уровне.

In [ ]:
EXPERIMENTS = [
    {
        "name": "hier_artist_album_rq_a512_b64_r3",
        "artist_k": 512,
        "album_k": 64,
        "residual_levels": 3,
        "residual_k": 512,
        "kmeans_iters": 20,
        "batch_size": 8192,
        "seed": 42,
    },
    # Optional heavier variant:
    # {
    #     "name": "hier_artist_album_rq_a1024_b64_r3",
    #     "artist_k": 1024,
    #     "album_k": 64,
    #     "residual_levels": 3,
    #     "residual_k": 512,
    #     "kmeans_iters": 20,
    #     "batch_size": 8192,
    #     "seed": 42,
    # },
]

RUN_SEQREC = True
EXPERIMENTS

## Load Item Table

Используем тот же train+holdout catalog, на котором `train_holdout` dVAE/RKMeans обычно строит финальные SIDs. Metadata берется из `item_metadata.parquet`, который уже создается основным notebook.

In [ ]:
train_items = pl.read_parquet(DATA_DIR / "dvae_train_items.parquet")
holdout_items = pl.read_parquet(DATA_DIR / "dvae_holdout_items.parquet")
metadata = pl.read_parquet(DATA_DIR / "item_metadata.parquet")

items = (
    pl.concat([train_items, holdout_items], how="vertical", rechunk=True)
    .select("item_id", "embed")
    .join(metadata.select("item_id", "artist_id", "album_id"), on="item_id", how="left")
)

item_ids = items["item_id"].to_list()
artist_ids = items["artist_id"].to_list()
album_ids = items["album_id"].to_list()
X_np = np.stack(items["embed"].to_list()).astype("float32")
X = torch.from_numpy(X_np).to(DEVICE)
X = F.normalize(X, dim=-1)

print("items:", len(item_ids), "dim:", X.shape[-1])
print("missing artists:", sum(x is None for x in artist_ids))
print("missing albums:", sum(x is None for x in album_ids))

def dense_int_labels(values):
    keys = sorted({v for v in values if v is not None}, key=str)
    mapping = {key: idx + 1 for idx, key in enumerate(keys)}
    return [mapping.get(v, 0) for v in values], mapping

artist_label, artist_label_map = dense_int_labels(artist_ids)
album_label, album_label_map = dense_int_labels(album_ids)
label_metadata_path = DATA_DIR / "item_metadata_raw_dense_labels.parquet"
pl.DataFrame({
    "item_id": item_ids,
    "artist_label": artist_label,
    "album_label": album_label,
}).write_parquet(label_metadata_path)
print("raw artist labels:", len(artist_label_map), "raw album labels:", len(album_label_map))

## K-Means Helpers

Минимальная torch-реализация k-means, чтобы не зависеть от sklearn. Для больших матриц distance считается батчами.

In [ ]:
@torch.no_grad()
def assign_labels(x, centroids, batch_size=8192):
    labels = torch.empty(x.shape[0], dtype=torch.long, device=x.device)
    c_norm = centroids.pow(2).sum(dim=1)[None, :]
    for start in range(0, x.shape[0], batch_size):
        end = min(start + batch_size, x.shape[0])
        xb = x[start:end]
        dists = xb.pow(2).sum(dim=1, keepdim=True) + c_norm - 2.0 * xb @ centroids.T
        labels[start:end] = dists.argmin(dim=1)
    return labels

@torch.no_grad()
def kmeans(x, k, *, num_iters=20, batch_size=8192, seed=42, verbose=False):
    if x.shape[0] == 0:
        raise ValueError("kmeans received an empty tensor")
    k = min(int(k), int(x.shape[0]))
    gen = torch.Generator(device=x.device)
    gen.manual_seed(seed)
    perm = torch.randperm(x.shape[0], device=x.device, generator=gen)
    centroids = x[perm[:k]].clone()

    for it in range(num_iters):
        labels = assign_labels(x, centroids, batch_size=batch_size)
        new_centroids = torch.zeros_like(centroids)
        counts = torch.bincount(labels, minlength=k).to(x.device)
        new_centroids.index_add_(0, labels, x)

        empty = counts == 0
        if empty.any():
            repl = torch.randint(0, x.shape[0], (int(empty.sum()),), device=x.device, generator=gen)
            new_centroids[empty] = x[repl]
            counts = counts.clone()
            counts[empty] = 1

        new_centroids = new_centroids / counts.clamp_min(1).float().unsqueeze(1)
        shift = (centroids - new_centroids).norm(dim=1).max().item()
        centroids = new_centroids
        if verbose:
            print(f"iter={it:02d} shift={shift:.6f}")
        if shift < 1e-4:
            break

    labels = assign_labels(x, centroids, batch_size=batch_size)
    return labels, centroids

def mean_vectors_by_key(keys, vectors):
    groups = defaultdict(list)
    for idx, key in enumerate(keys):
        if key is not None:
            groups[key].append(idx)
    out_keys = list(groups)
    out_vecs = torch.stack([vectors[groups[key]].mean(dim=0) for key in out_keys])
    out_vecs = F.normalize(out_vecs, dim=-1)
    return out_keys, out_vecs, groups

## Build Hierarchical SIDs

Алгоритм:

1. Средний embedding на artist → k-means → `sid[0]`.
2. Средний embedding на album внутри каждого artist cluster → локальный k-means → `sid[1]`.
3. Вычитаем artist/album centroids из item embedding.
4. На остатке делаем обычный residual k-means → `sid[2:]`.

In [ ]:
@torch.no_grad()
def build_hierarchical_sids(exp):
    seed = int(exp["seed"])
    batch_size = int(exp["batch_size"])
    num_iters = int(exp["kmeans_iters"])

    # 1) Artist clusters from mean artist embeddings. Label 0 is unknown.
    artist_keys, artist_vecs, _ = mean_vectors_by_key(artist_ids, X)
    artist_labels_raw, artist_centroids_raw = kmeans(
        artist_vecs,
        exp["artist_k"],
        num_iters=num_iters,
        batch_size=batch_size,
        seed=seed,
        verbose=True,
    )
    raw_artist_to_cluster = {
        key: int(label.item()) + 1
        for key, label in zip(artist_keys, artist_labels_raw, strict=True)
    }
    artist_centroids = torch.cat([
        torch.zeros(1, X.shape[1], device=X.device),
        artist_centroids_raw,
    ], dim=0)
    artist_code = torch.tensor(
        [raw_artist_to_cluster.get(a, 0) for a in artist_ids],
        dtype=torch.long,
        device=X.device,
    )

    # 2) Album clusters inside each artist cluster. Label 0 is unknown.
    album_groups = defaultdict(list)
    for idx, (artist_cluster, album_id) in enumerate(zip(artist_code.tolist(), album_ids, strict=True)):
        if album_id is not None:
            album_groups[(artist_cluster, album_id)].append(idx)

    albums_by_artist_cluster = defaultdict(list)
    album_vec_by_key = {}
    for key, idxs in album_groups.items():
        artist_cluster, _ = key
        album_vec = F.normalize(X[idxs].mean(dim=0, keepdim=True), dim=-1).squeeze(0)
        album_vec_by_key[key] = album_vec
        albums_by_artist_cluster[artist_cluster].append(key)

    album_code_by_key = {}
    album_centroid_by_local_key = {}
    for artist_cluster, keys in sorted(albums_by_artist_cluster.items()):
        vecs = torch.stack([album_vec_by_key[key] for key in keys])
        labels, centroids = kmeans(
            vecs,
            exp["album_k"],
            num_iters=num_iters,
            batch_size=batch_size,
            seed=seed + int(artist_cluster),
            verbose=False,
        )
        for key, label in zip(keys, labels.tolist(), strict=True):
            local_code = int(label) + 1
            album_code_by_key[key] = local_code
        for local_code, centroid in enumerate(centroids, start=1):
            album_centroid_by_local_key[(artist_cluster, local_code)] = centroid

    album_code_list = []
    album_centroid_list = []
    zero = torch.zeros(X.shape[1], device=X.device)
    for artist_cluster, album_id in zip(artist_code.tolist(), album_ids, strict=True):
        local_code = album_code_by_key.get((artist_cluster, album_id), 0)
        album_code_list.append(local_code)
        album_centroid_list.append(album_centroid_by_local_key.get((artist_cluster, local_code), zero))

    album_code = torch.tensor(album_code_list, dtype=torch.long, device=X.device)
    album_centroids_per_item = torch.stack(album_centroid_list)

    # 3) Residual k-means levels.
    residual = X - artist_centroids[artist_code] - album_centroids_per_item
    residual_codes = []
    residual_centroids = []
    for level in range(int(exp["residual_levels"])):
        labels, centroids = kmeans(
            residual,
            exp["residual_k"],
            num_iters=num_iters,
            batch_size=batch_size,
            seed=seed + 1000 + level,
            verbose=True,
        )
        residual_codes.append(labels)
        residual_centroids.append(centroids)
        residual = residual - centroids[labels]

    codes = torch.stack([artist_code, album_code, *residual_codes], dim=1).cpu().numpy().astype("int64")
    sid_lists = [row.tolist() for row in codes]
    length = codes.shape[1]

    return pl.DataFrame({
        "item_id": item_ids,
        "sid": sid_lists,
        "length": [length] * len(item_ids),
    }), {
        "name": exp["name"],
        "artist_k_requested": exp["artist_k"],
        "artist_k_effective": int(artist_centroids_raw.shape[0]),
        "album_k_per_artist_requested": exp["album_k"],
        "residual_levels": exp["residual_levels"],
        "residual_k": exp["residual_k"],
        "sid_length": int(length),
        "num_items": len(item_ids),
        "num_raw_artists": len(artist_keys),
        "num_raw_album_artist_pairs": len(album_groups),
    }

## Build SIDs and Structural Metrics

Для каждого experiment пишем:

- `results/RQ_album_artist_anchor/<name>/sids.parquet`
- `sid_metrics.json`
- `prefix_purity.json`

In [ ]:
for exp in EXPERIMENTS:
    out_dir = RESULTS_DIR / exp["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n===", exp["name"], "===")

    sids, sid_metrics = build_hierarchical_sids(exp)
    sids_path = out_dir / "sids.parquet"
    sids.write_parquet(sids_path)
    (out_dir / "sid_metrics.json").write_text(json.dumps(sid_metrics, indent=2))
    print(json.dumps(sid_metrics, indent=2))

    run([
        PY, "-m", "scripts.RQ_album_artist_anchor.analyze_prefix_purity",
        "--sids", sids_path,
        "--metadata", label_metadata_path,
        "--output", out_dir / "prefix_purity.json",
        "--artist-col", "artist_label",
        "--album-col", "album_label",
    ])

## Run Seqrec

Создаем seqrec config на базе `seqrec_fixed.yaml`, меняя только `semantic_ids_path` и `summary_json`. Метрики будут `@10`, `@50`, `@100`.

In [ ]:
base_seqrec_cfg = yaml.safe_load((CONFIG_DIR / "seqrec_fixed.yaml").read_text())

if RUN_SEQREC:
    for exp in EXPERIMENTS:
        name = exp["name"]
        out_dir = RESULTS_DIR / name
        cfg = copy.deepcopy(base_seqrec_cfg)
        cfg["semantic_ids_path"] = f"./results/RQ_album_artist_anchor/{name}/sids.parquet"
        cfg["summary_json"] = f"./results/RQ_album_artist_anchor/{name}/seqrec_summary.json"
        cfg_path = CONFIG_DIR / f"seqrec_{name}.yaml"
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
        run([PY, "-m", "scripts.train_seqrec", "--config", cfg_path])
else:
    print("RUN_SEQREC=False, skipping seqrec training/eval")

## Compact Comparison

Читаем новые результаты и, если уже есть предыдущие runs, добавляем их в ту же таблицу.

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text())

methods = ["fixed", "original", "aux", "prefix"] + [exp["name"] for exp in EXPERIMENTS]
rows = []
for method in methods:
    method_dir = RESULTS_DIR / method
    seqrec = load_json(method_dir / "seqrec_summary.json")
    purity = load_json(method_dir / "prefix_purity.json")
    metrics = seqrec.get("metrics", {}) if seqrec else {}
    row = {
        "method": method,
        "recall@10": metrics.get("recall@10"),
        "recall@50": metrics.get("recall@50"),
        "recall@100": metrics.get("recall@100"),
        "ndcg@10": metrics.get("ndcg@10"),
        "ndcg@50": metrics.get("ndcg@50"),
        "ndcg@100": metrics.get("ndcg@100"),
        "coverage@100": metrics.get("coverage@100"),
        "head_recall@10": metrics.get("head_recall@10"),
        "tail_recall@10": metrics.get("tail_recall@10"),
        "artist_purity@1": purity.get("artist_prefix_purity", {}).get("1", {}).get("purity") if purity else None,
        "album_purity@2": purity.get("album_prefix_purity", {}).get("2", {}).get("purity") if purity else None,
        "mean_sid_length": purity.get("mean_sid_length") if purity else None,
        "excess_item_collisions": purity.get("excess_item_collisions") if purity else None,
    }
    rows.append(row)

comparison_path = RESULTS_DIR / "hierarchical_extra_comparison.json"
comparison_path.write_text(json.dumps(rows, indent=2))
rows